<a href="https://colab.research.google.com/github/affanahmed373/Invoice-Agent/blob/main/vector_storage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community
!pip install pypdf
!pip install -U langchain-text-splitters
!pip install qdrant-client
!pip install sentence-transformers

In [ ]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from qdrant_client.http.models import Distance, VectorParams
import hashlib

In [ ]:
collection_name = "pdf_document_embeddings"
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
QDRANT_HOST = userdata.get('QDRANT_HOST')
embedding_model_name = "all-MiniLM-L6-v2" # A good, lightweight general-purpose model
embedding_dimension = 384
collection_name = "pdf_document_embeddings"


In [ ]:


client = QdrantClient(
    url=QDRANT_HOST,
    api_key=QDRANT_API_KEY,
)

print("\n--- Performing Qdrant Connection Check ---")

try:
    # Attempt to list collections as a simple connection test
    collections = client.get_collections()
    print("Successfully connected to Qdrant! Existing collections:")
    for collection in collections.collections:
        print(f"- {collection.name}")
except Exception as e:
    print(f"Error connecting to Qdrant or listing collections: {e}")
    print("Please check your QDRANT_HOST, QDRANT_API_KEY, and network connection, and ensure secrets are formatted correctly (no extra quotes).")

print("--- Connection Check Complete ---")

print("Qdrant client initialized successfully.")



--- Performing Qdrant Connection Check ---
Successfully connected to Qdrant! Existing collections:
- pdf_document_embeddings
--- Connection Check Complete ---
Qdrant client initialized successfully.


In [ ]:
#pdf_name ='326113641__0.pdf'
#pdf_name ='InvoiceRE-202659666.pdf'
pdf_name ='SalesInvoice.pdf'

In [ ]:
reader = PdfReader(pdf_name)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True,
)

texts = []
for i, page in enumerate(reader.pages):
    page_text = page.extract_text()
    if page_text: # Only process pages that actually have text
        # create_documents expects a list of strings, and optional metadatas list
        # The metadatas will be added to each Document object created by the splitter.
        chunks_for_page = text_splitter.create_documents(
            [page_text],
            metadatas=[{"source": pdf_name, "page_number": i + 1}]
        )
        texts.extend(chunks_for_page)

print(f"Split into {len(texts)} documents from {len(reader.pages)} pages.")
if texts:
    print("First document content (truncated): ")
    print(texts[0].page_content[:500])
    print("First document metadata:", texts[0].metadata)
else:
    print("No text or chunks were extracted from the PDF.")

Split into 6 documents from 2 pages.
First document content (truncated): 
Facture
Invoive No   
2607361
Billing Address
ZAKI ASIAN FOODS
         
ZAKI 
EXOTIC CITY SRL
Universitätsstr ,40
Avenue de l'expansion 1
Duisburg, 47051
ALLEUR
Bill-to Customer 
2020564
 Enterprise No.
  
BE 0882.540.147
Enterprise No.
 
  Phone No.
VATRegistrationNo
352116861
No.Telephone No.
 E-mail   
             
info@exoticcity.be
Invoice No.
2607361
  HomePage 
https://exoticcity.be
 Document Date
03-05-2026
  Bank
BELFIUS
Posting Date
03-05-2026
 SWIFT Code
GKCCBEBB
Due Date
10-05-2026
First document metadata: {'source': 'SalesInvoice.pdf', 'page_number': 1, 'start_index': 0}


In [ ]:
print(f"Loading Sentence Transformer model: {embedding_model_name}...")
model = SentenceTransformer(embedding_model_name)
print("Model loaded.")

# Generate embeddings for the text chunks
print("Generating embeddings for text chunks...")
# 'texts' is a list of Document objects from Langchain, extract page_content
text_contents = [doc.page_content for doc in texts]
embeddings = model.encode(text_contents, show_progress_bar=True)
print(f"Generated {len(embeddings)} embeddings of dimension {embeddings.shape[1]}.")

Loading Sentence Transformer model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded.
Generating embeddings for text chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated 6 embeddings of dimension 384.


In [ ]:

print(f"Preparing points for upserting into collection '{collection_name}'...")

points = []
# Iterate over both embeddings and the original Document objects from 'texts'
for i, (embedding, doc) in enumerate(zip(embeddings, texts)):
    # The payload should contain the text content and all metadata
    payload = {"text": doc.page_content}
    # Add all metadata fields from the Langchain Document to the payload
    # This will include 'source', 'page_number', and 'start_index'
    payload.update(doc.metadata)


    point_id = int(hashlib.sha256(doc.page_content.encode('utf-8')).hexdigest(), 16) % (10**18) # Generate a large integer ID

    points.append({
        "id": point_id, # Using a unique ID based on content hash
        "vector": embedding.tolist(), # Convert numpy array to list
        "payload": payload
    })

print(f"Upserting {len(points)} new points into collection '{collection_name}'...")
client.upsert(
    collection_name=collection_name,
    wait=True, # Wait for the operation to complete
    points=points
)
print(f"Successfully added {len(points)} new points to Qdrant collection '{collection_name}'.")
print("Example payload for a new point (truncated):")
if points:
    display(points[0]['payload'])
else:
    print("No points were created.")

Preparing points for upserting into collection 'pdf_document_embeddings'...
Upserting 6 new points into collection 'pdf_document_embeddings'...
Successfully added 6 new points to Qdrant collection 'pdf_document_embeddings'.
Example payload for a new point (truncated):


{'text': "Facture\nInvoive No   \n2607361\nBilling Address\nZAKI ASIAN FOODS\n         \nZAKI \nEXOTIC CITY SRL\nUniversitätsstr ,40\nAvenue de l'expansion 1\nDuisburg, 47051\nALLEUR\nBill-to Customer \n2020564\n Enterprise No.\n  \nBE 0882.540.147\nEnterprise No.\n \n  Phone No.\nVATRegistrationNo\n352116861\nNo.Telephone No.\n E-mail   \n             \ninfo@exoticcity.be\nInvoice No.\n2607361\n  HomePage \nhttps://exoticcity.be\n Document Date\n03-05-2026\n  Bank\nBELFIUS\nPosting Date\n03-05-2026\n SWIFT Code\nGKCCBEBB\nDue Date\n10-05-2026\nPayment Term\nNetto 7 dagen\n User Code \nFT-ADNAN\nCondition Paiment\nPartner Type\n IBAN\nBE71 0688 9436 3669\nCondition Livraison\nDAP\n \nVendeur\nAQ-W10\nDriver Name\nDIALLO \nVehicle No.\n1-SRL-457\nQTY\nUOM\nDiscount\nAmount \nIncluding \nVAT Line\nkindly return the pallet to avoid monthly charges 12€. Thank you for your \ncooperation.\nnous vous prions de bien vouloir nous retourner la palette afin d'éviter des frais \nmensuels 12€. Nou\

use with caution

In [ ]:
from qdrant_client import models

# To delete all points in the collection, use an empty `must` list in the filter
filter_condition = models.Filter(must=[])

client.delete(
    collection_name=collection_name,
    wait=True, # Wait for the operation to complete
    points_selector=models.FilterSelector(filter=filter_condition)
)
print(f"Deleted all points from collection: '{collection_name}'")

Deleted all points from collection: 'pdf_document_embeddings'
